# SimGuard -- Stage 1: Prophet Burn-Rate Forecasting

Forecasts each SIM's **cumulative data usage** within its current billing cycle
using Facebook Prophet, and flags SIMs projected to exhaust their bundle before
the cycle ends. This is a **parallel Stage 1 signal** -- independent of Isolation
Forest, not downstream of it.

| # | Cell | Purpose |
|---|------|---------|
| 1 | Imports & installs | prophet, pandas, numpy, matplotlib, mlflow |
| 2 | Paths, seed, MLflow | SEED, dirs, new experiment |
| 3 | Load raw data | daily\_agg CSV + labels CSV (kept separate) |
| 4 | Bundle map | tier -> bundle\_mb lookup |
| 5 | Billing cycle | **fixed 30-day** billing\_cycle\_num per row |
| 6 | Forecast function | per-(SIM, cycle) Prophet fit on cumulative MB |
| 7 | Extract final-day prediction | yhat on last cycle day |
| 8 | Depletion signal | projected\_overage\_mb, depletion\_flag, days\_to\_depletion |
| 9 | Loop execution | subsample toggle, runtime estimate, progress reporting |
| 10 | Ghost/inactive caveat | cold-start behaviour report (labels joined here only) |
| 11 | Build feature table | one row per sim\_id per date |
| 12 | Save | data/prophet\_features.parquet |
| 13 | Evaluation | depletion\_flag vs misuse labels |
| 14 | Example plots | 3 SIMs with forecast overlaid |

> **Colab users:** upload `ml-service/` and run `import os; os.chdir('ml-service')`
> before executing. Adjust `DATA_DIR` / `ARTIFACT_DIR` in Cell 2 if needed.

## Cell 1 -- Imports & Installs
Prophet's package name is `prophet` (formerly `fbprophet`).
The install can take 1-2 minutes in Colab.

In [1]:
# Cell 1: Imports & Installs
# Adjust DATA_DIR / ARTIFACT_DIR in Cell 2 if your Colab upload structure differs.

!pip install -q prophet pandas numpy matplotlib mlflow pyarrow joblib

import os, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')   # headless; remove this line if running interactively
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import joblib
import mlflow

from prophet import Prophet
from pandas.tseries.offsets import MonthEnd

warnings.filterwarnings('ignore')   # Prophet prints many convergence notes
import logging
logging.getLogger('prophet').setLevel(logging.WARNING)
logging.getLogger('cmdstanpy').setLevel(logging.WARNING)

print('[OK] Imports complete.')


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
C:\Users\ADMIN\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Importing plotly failed. Interactive plots will not work.


[OK] Imports complete.


## Cell 2 -- Seed, Paths & MLflow Setup
New MLflow experiment: `simguard-stage1-prophet-burnrate`.

In [2]:
# Cell 2: Seed, Paths & MLflow Setup

SEED = 42
np.random.seed(SEED)

#  Paths (relative to ml-service/) 
# Colab users: adjust if your upload placed files elsewhere.
DATA_DIR     = 'data'
ARTIFACT_DIR = 'artifacts'
os.makedirs(ARTIFACT_DIR, exist_ok=True)

# Auto-detect if running from notebooks/ subdirectory 
if not os.path.isdir(DATA_DIR) and os.path.isdir(os.path.join('..', DATA_DIR)):
    os.chdir('..')
    print(f'  CWD adjusted to: {os.getcwd()}')

assert os.path.isdir(DATA_DIR), (
    f'DATA_DIR={DATA_DIR!r} not found from CWD={os.getcwd()!r}. '
    f'If running in Colab, run: import os; os.chdir("ml-service") first.'
)

#  MLflow 
os.environ.setdefault('MLFLOW_ALLOW_FILE_STORE', 'true')
MLFLOW_DB = 'sqlite:///mlruns.db'
mlflow.set_tracking_uri(MLFLOW_DB)
mlflow.set_experiment('simguard-stage1-prophet-burnrate')

print(f'SEED         : {SEED}')
print(f'DATA_DIR     : {os.path.abspath(DATA_DIR)}')
print(f'ARTIFACT_DIR : {os.path.abspath(ARTIFACT_DIR)}')
print(f'MLflow URI   : {mlflow.get_tracking_uri()}')
print(f'MLflow exp   : simguard-stage1-prophet-burnrate')
print('[OK] Setup complete.')

  CWD adjusted to: c:\Users\ADMIN\Desktop\projects\simguard\ml-service
SEED         : 42
DATA_DIR     : c:\Users\ADMIN\Desktop\projects\simguard\ml-service\data
ARTIFACT_DIR : c:\Users\ADMIN\Desktop\projects\simguard\ml-service\artifacts
MLflow URI   : sqlite:///mlruns.db
MLflow exp   : simguard-stage1-prophet-burnrate
[OK] Setup complete.


## Cell 3 -- Load Raw Data

Loads `synthetic_cdr_daily_agg.csv` -- the **raw** per-SIM per-day aggregates
with real calendar dates and unscaled `total_mb`. Prophet needs this, not the
scaled/feature-engineered parquet files.

`synthetic_cdr_labels.csv` is loaded separately and kept completely apart
from every cell that fits or applies Prophet. It is only joined at Cell 13
(evaluation) and Cell 10 (caveat report).

**Labels schema:** `sim_id`, `category`, `start_day`, `end_day`, `duration_days`
where `start_day` / `end_day` are 0-indexed integer offsets from 2025-01-01
(the simulation start date).

In [3]:
# Cell 3: Load Raw Data

SIM_START = pd.Timestamp('2025-01-01')   # global day-0 for offset->date mapping

#  Daily aggregates (Prophet input) 
AGG_PATH = os.path.join(DATA_DIR, 'synthetic_cdr_daily_agg.csv')
agg = pd.read_csv(AGG_PATH, parse_dates=['date'])
agg['date'] = pd.to_datetime(agg['date'])

print(f'Daily agg shape   : {agg.shape}')
print(f'Date range        : {agg.date.min().date()}  ->  {agg.date.max().date()}')
print(f'Unique SIMs       : {agg.sim_id.nunique():,}')
print(f'Columns           : {list(agg.columns)}')

# Labels (kept separate  only joined at evaluation step) 
LBL_PATH = os.path.join(DATA_DIR, 'synthetic_cdr_labels.csv')
labels_raw = pd.read_csv(LBL_PATH)

# Convert integer offsets to actual dates
labels_raw['start_date'] = SIM_START + pd.to_timedelta(labels_raw['start_day'], unit='D')
labels_raw['end_date']   = SIM_START + pd.to_timedelta(labels_raw['end_day'],   unit='D')

print(f'\nLabels shape      : {labels_raw.shape}')
print(f'Label categories  : {labels_raw.category.value_counts().to_dict()}')
print('\n[OK] Data loaded. Labels will not be used until Cell 10 (caveat) and Cell 13 (eval).')

Daily agg shape   : (90000, 20)
Date range        : 2025-01-01  ->  2025-03-31
Unique SIMs       : 1,000
Columns           : ['sim_id', 'date', 'total_download_mb', 'total_upload_mb', 'data_session_count', 'data_duration_s', 'hotspot_sessions', 'public_apn_sessions', 'unique_towers', 'geo_flag_count', 'peak_mb', 'offpeak_mb', 'total_mb', 'roaming_sessions', 'billed_mb', 'tier', 'voice_minutes_used', 'sms_count', 'voice_minutes_remaining', 'sms_remaining']

Labels shape      : (175, 7)
Label categories  : {'unauthorized_sharing': 69, 'ghost_usage': 43, 'data_reselling': 42, 'inactive_billing': 21}

[OK] Data loaded. Labels will not be used until Cell 10 (caveat) and Cell 13 (eval).


## Cell 4 -- Tier Bundle Map
Maps each tier to its monthly bundle allocation.
Values match the synthetic data generator's tier parameters.

In [4]:
# Cell 4: Tier Bundle Map

TIER_BUNDLE_GB = {
    'field':        100,
    'manager':       60,
    'professional':  40,
    'support':       20,
}
TIER_BUNDLE_MB = {t: gb * 1024 for t, gb in TIER_BUNDLE_GB.items()}

# Add bundle_mb column to the daily agg
agg['bundle_mb'] = agg['tier'].map(TIER_BUNDLE_MB)

assert agg['bundle_mb'].isna().sum() == 0, 'Unknown tier found -- update TIER_BUNDLE_GB'

print('Tier -> bundle mapping:')
for tier, gb in TIER_BUNDLE_GB.items():
    mb = TIER_BUNDLE_MB[tier]
    n  = (agg.tier == tier).sum()
    print(f'  {tier:<15}: {gb:>4} GB  ({mb:>6,} MB)   rows in agg: {n:,}')
print('\n[OK] bundle_mb column added.')

Tier -> bundle mapping:
  field          :  100 GB  (102,400 MB)   rows in agg: 5,670
  manager        :   60 GB  (61,440 MB)   rows in agg: 29,610
  professional   :   40 GB  (40,960 MB)   rows in agg: 41,220
  support        :   20 GB  (20,480 MB)   rows in agg: 13,500

[OK] bundle_mb column added.


## Cell 5 -- Billing Cycle Column (fixed 30-day cycles)

Assigns each row to a **fixed 30-day billing cycle** counted from the simulation
start date (2025-01-01), rather than calendar months.

Calendar-month cycles are incomparable: January has 31 days, February 28.
A SIM depleting on day 28 of a 31-day cycle is at different risk than one
depleting on day 28 of a 28-day cycle. Fixed 30-day windows make
`days_to_depletion` and `projected_overage_mb` comparable across all cycles.

Columns added:
- `day_offset` -- integer days since 2025-01-01
- `billing_cycle_num` -- 0, 1, 2 ... (cycle index)
- `cycle_start_date` -- first day of the 30-day window
- `cycle_end_date` -- last day of the 30-day window (start + 29 days)

Prints the day-count per (sim\_id, cycle) group to confirm uniformity.

In [5]:
# Cell 5: Billing Cycle Column (fixed 30-day cycles)

agg['day_offset']       = (agg['date'] - SIM_START).dt.days
agg['billing_cycle_num'] = agg['day_offset'] // 30   # 0, 1, 2 for a 90-day window
agg['cycle_start_date'] = SIM_START + pd.to_timedelta(agg['billing_cycle_num'] * 30, unit='D')
agg['cycle_end_date']   = agg['cycle_start_date'] + pd.Timedelta(days=29)

cycle_sizes = agg.groupby(['sim_id', 'billing_cycle_num']).size()
print('Day count per (sim_id, billing_cycle_num) group:')
print(cycle_sizes.value_counts().sort_index().to_string())
print()
print('Cycle start/end dates:')
for cyc_num in sorted(agg['billing_cycle_num'].unique()):
    sample = agg[agg['billing_cycle_num'] == cyc_num].iloc[0]
    print(f'  billing_cycle_num={cyc_num}  '
          f'{sample["cycle_start_date"].date()} -> {sample["cycle_end_date"].date()}')
print(f'\nTotal (sim_id, billing_cycle_num) groups: '
      f'{agg.groupby(["sim_id","billing_cycle_num"]).ngroups:,}')
print('[OK] Fixed 30-day billing cycles added.')

Day count per (sim_id, billing_cycle_num) group:
30    3000

Cycle start/end dates:
  billing_cycle_num=0  2025-01-01 -> 2025-01-30
  billing_cycle_num=1  2025-01-31 -> 2025-03-01
  billing_cycle_num=2  2025-03-02 -> 2025-03-31

Total (sim_id, billing_cycle_num) groups: 3,000
[OK] Fixed 30-day billing cycles added.


## Cell 6 -- Per-SIM Per-Cycle Prophet Forecast Function

Defines the core forecasting logic applied inside the loop (Cell 9).

**Cumulative usage:** Prophet is fit on the SIM's *running total* of `total_mb`
within the cycle, not the daily values. A bundle is depleted when the running
total crosses `bundle_mb`, so forecasting the cumulative trajectory is the
natural formulation.

**Cycle end date:** `cycle_end_date` is read directly from the `sim_df` column
(a fixed 30-day window computed in Cell 5). This replaces the previous
`MonthEnd(1)` calculation, which gave variable-length cycles (28/30/31 days).

**Cold-start skip:** Groups with fewer than 7 elapsed days are skipped.
This avoids fitting Prophet on too little history. The first 7 days of every
cycle have `depletion_flag = NaN` (not imputed).

**Prophet settings:**
- `daily_seasonality=False` -- 30-day cycles offer insufficient data for daily pattern
- `weekly_seasonality=True` -- weekday/weekend pattern can matter for data usage
- `yearly_seasonality=False` -- only 3 months of data, not enough for annual fit

In [6]:
# Cell 6: Per-SIM Per-Cycle Forecast Function

MIN_HISTORY_DAYS = 7   # cold-start threshold


def forecast_cycle(sim_df, bundle_mb):
    '''
    Fit Prophet on cumulative total_mb within a fixed 30-day billing cycle
    and forecast to the last day of that cycle.

    Parameters
    ----------
    sim_df    : DataFrame with columns [date, total_mb, cycle_end_date], sorted
                by date, all rows within ONE (sim_id, billing_cycle_num).
    bundle_mb : float -- the SIM tier bundle in MB.

    Returns
    -------
    dict with keys:
        forecast_df          : Prophet forecast DataFrame (ds, yhat, yhat_lower, yhat_upper)
        predicted_final_cum  : yhat on cycle_end_date
        projected_overage_mb : predicted_final_cum - bundle_mb
        depletion_flag       : bool (projected to exceed bundle)
        days_to_depletion    : int or NaN (first forecast day yhat crosses bundle_mb)
    '''
    sim_df = sim_df.sort_values('date').reset_index(drop=True)
    n_elapsed = len(sim_df)

    if n_elapsed < MIN_HISTORY_DAYS:
        return None   # cold-start: not enough history

    # Cumulative usage within cycle
    cum_mb = sim_df['total_mb'].cumsum().values

    prophet_df = pd.DataFrame({
        'ds': sim_df['date'],
        'y':  cum_mb,
    })

    # Last day of the fixed 30-day billing cycle (from the column, not MonthEnd)
    cycle_end = pd.Timestamp(sim_df['cycle_end_date'].iloc[0])

    # Future dates: from the day after last observed to cycle end
    last_obs = sim_df['date'].iloc[-1]
    future_dates = pd.date_range(start=last_obs + pd.Timedelta(days=1),
                                  end=cycle_end, freq='D')

    m = Prophet(
        daily_seasonality=False,
        weekly_seasonality=True,
        yearly_seasonality=False,
        changepoint_prior_scale=0.3,   # allow trend to flex for short cycles
    )
    m.fit(prophet_df)

    if len(future_dates) == 0:
        # Already at last day of cycle -- use fitted values directly
        forecast = m.predict(prophet_df[['ds']])
        pred_final = float(forecast['yhat'].iloc[-1])
        full_forecast = forecast
    else:
        future_df = pd.DataFrame({'ds': pd.concat(
            [prophet_df['ds'], pd.Series(future_dates)],
            ignore_index=True)})
        forecast = m.predict(future_df)
        pred_final = float(forecast[forecast['ds'] == cycle_end]['yhat'].values[0])
        full_forecast = forecast

    overage_mb   = pred_final - bundle_mb
    dep_flag     = bool(overage_mb > 0)

    # First forecast day where cumulative yhat crosses bundle_mb
    cross = full_forecast[full_forecast['yhat'] >= bundle_mb]
    if len(cross) > 0:
        first_cross_date = cross['ds'].iloc[0]
        days_to_dep      = int((first_cross_date - last_obs).days)
    else:
        days_to_dep = float('nan')

    return {
        'forecast_df':          full_forecast,
        'predicted_final_cum':  pred_final,
        'projected_overage_mb': overage_mb,
        'depletion_flag':       dep_flag,
        'days_to_depletion':    days_to_dep,
        'cycle_end':            cycle_end,
        'last_obs_date':        last_obs,
    }

print(f'[OK] forecast_cycle() defined.  Cold-start threshold: {MIN_HISTORY_DAYS} days.')
print('  cycle_end is read from sim_df["cycle_end_date"] -- fixed 30-day window.')

[OK] forecast_cycle() defined.  Cold-start threshold: 7 days.
  cycle_end is read from sim_df["cycle_end_date"] -- fixed 30-day window.


## Cell 7 -- Forecast Extraction Notes
Explains the prediction extraction logic already embedded in `forecast_cycle`.
Kept as a separate cell so it can be re-read without re-running the loop.

In [7]:
# Cell 7: Forecast Extraction Notes (documentation cell, nothing to run)

print('Prediction extraction (inside forecast_cycle):')  
print('  predicted_final_cum  = yhat on the last calendar day of the cycle')
print('  projected_overage_mb = predicted_final_cum - bundle_mb')
print('  depletion_flag       = projected_overage_mb > 0')
print('  days_to_depletion    = first forecast day where cumulative yhat >= bundle_mb')
print('                         (NaN if bundle is never projected to be crossed)')
print()
print('Cold-start window (first 7 days of any cycle):')
print('  depletion_flag = NaN, projected_overage_mb = NaN, days_to_depletion = NaN')
print('  This is a known limitation analogous to ghost_usage / inactive_billing')
print('  SIMs having flat near-zero usage -- Prophet needs real trend to extrapolate.')

Prediction extraction (inside forecast_cycle):
  predicted_final_cum  = yhat on the last calendar day of the cycle
  projected_overage_mb = predicted_final_cum - bundle_mb
  depletion_flag       = projected_overage_mb > 0
  days_to_depletion    = first forecast day where cumulative yhat >= bundle_mb
                         (NaN if bundle is never projected to be crossed)

Cold-start window (first 7 days of any cycle):
  depletion_flag = NaN, projected_overage_mb = NaN, days_to_depletion = NaN
  This is a known limitation analogous to ghost_usage / inactive_billing
  SIMs having flat near-zero usage -- Prophet needs real trend to extrapolate.


## Cell 8 -- Depletion Signal Definition
Defines the output columns that will be computed for each (SIM, cycle) group.

In [8]:
# Cell 8: Depletion Signal Definition

OUTPUT_COLS = [
    'sim_id',
    'date',
    'billing_cycle_num',
    'depletion_flag',
    'projected_overage_mb',
    'days_to_depletion',
]

print('Output columns per (sim_id, date) row:')
for c in OUTPUT_COLS:
    print(f'  {c}')
print()
print('NaN policy:')
print('  First 7 days of each cycle (cold-start): all three signal cols = NaN.')
print('  Do NOT impute. Downstream models must handle NaN explicitly.')

Output columns per (sim_id, date) row:
  sim_id
  date
  billing_cycle_num
  depletion_flag
  projected_overage_mb
  days_to_depletion

NaN policy:
  First 7 days of each cycle (cold-start): all three signal cols = NaN.
  Do NOT impute. Downstream models must handle NaN explicitly.


## Cell 9 -- Forecast Loop

Iterates over every `(sim_id, billing_cycle_num)` group.
Before starting, times a single fit on a representative group to estimate
total runtime.

**Subsample toggle:** Set `SUBSAMPLE_SIMS` to an integer (e.g. `50`) to run
on a subset first. Default is `50` for a quick sanity-check. Set to `None`
for the full 3,000-group run (~110 min).

> **Runtime note:** Prophet fits are independent Stan MCMC/VI calls.
> With `algorithm='Newton'` (L-BFGS), each fit takes ~2 s.
> 3,000 groups x ~2 s ~ 100 minutes. The estimate is printed before the loop
> starts so you can decide whether to run in full or sub-sample.

Progress is printed every 50 iterations.
All fit errors are captured and saved to `artifacts/prophet_fit_errors.csv`.

In [9]:
# Cell 9: Forecast Loop

#  Subsample toggle (FIX 3) 
# Set to an integer (e.g. 50) to test on a subset; None = full run.
SUBSAMPLE_SIMS = None

if SUBSAMPLE_SIMS is not None:
    sample_sim_ids = agg['sim_id'].drop_duplicates().sample(
        n=SUBSAMPLE_SIMS, random_state=SEED)
    agg_loop = agg[agg['sim_id'].isin(sample_sim_ids)]
    print(f'SUBSAMPLE MODE: running on {SUBSAMPLE_SIMS} SIMs only.')
    print(f'  Set SUBSAMPLE_SIMS=None for the full run.')
else:
    agg_loop = agg
    print('FULL MODE: running on all SIMs.')

groups   = list(agg_loop.groupby(['sim_id', 'billing_cycle_num']))
n_groups = len(groups)
print(f'Total (sim_id, billing_cycle_num) groups: {n_groups:,}')

#  Estimate runtime from one sample fit 
print('Timing one Prophet fit to estimate total runtime ...')
sample_key, sample_df = groups[0]
sample_bundle = float(sample_df['bundle_mb'].iloc[0])
t0 = time.time()
_ = forecast_cycle(sample_df, sample_bundle)
single_fit_s = time.time() - t0
estimated_total_s   = single_fit_s * n_groups
estimated_total_min = estimated_total_s / 60
print(f'  Single fit time : {single_fit_s:.2f} s')
print(f'  Estimated total : {estimated_total_s:.0f} s  ({estimated_total_min:.1f} min)')
if estimated_total_min > 30:
    print(f'  [!] > 30 min -- consider keeping SUBSAMPLE_SIMS set for exploration.')
print()

#  Main loop 
print('Starting forecast loop ...')
t_loop_start = time.time()

results          = []   # one dict per (sim_id, cycle)
cold_start_count = 0
error_count      = 0
error_log        = []   # FIX 4: capture every error

for i, ((sim_id, cycle), grp) in enumerate(groups):
    bundle_mb = float(grp['bundle_mb'].iloc[0])

    try:
        res = forecast_cycle(grp, bundle_mb)
    except Exception as e:
        error_count += 1
        error_log.append({'sim_id': sim_id, 'cycle': str(cycle), 'error': str(e)})
        res = None

    if res is None:
        cold_start_count += 1
        results.append({
            'sim_id':              sim_id,
            'billing_cycle_num':   str(cycle),
            'depletion_flag':       float('nan'),
            'projected_overage_mb': float('nan'),
            'days_to_depletion':    float('nan'),
            'predicted_final_cum':  float('nan'),
            'forecast_df':          None,
            'last_obs_date':        None,
        })
    else:
        results.append({
            'sim_id':              sim_id,
            'billing_cycle_num':   str(cycle),
            'depletion_flag':       res['depletion_flag'],
            'projected_overage_mb': res['projected_overage_mb'],
            'days_to_depletion':    res['days_to_depletion'],
            'predicted_final_cum':  res['predicted_final_cum'],
            'forecast_df':          res['forecast_df'],
            'last_obs_date':        res['last_obs_date'],
        })

    if (i + 1) % 50 == 0 or (i + 1) == n_groups:
        elapsed = time.time() - t_loop_start
        rate    = (i + 1) / elapsed
        eta     = (n_groups - i - 1) / rate if rate > 0 else 0
        print(f'  [{i+1:>4}/{n_groups}]  elapsed={elapsed:.0f}s  '
              f'rate={rate:.2f}/s  ETA={eta:.0f}s')

total_loop_s = time.time() - t_loop_start
print(f'\nLoop complete in {total_loop_s:.0f} s ({total_loop_s/60:.1f} min)')
print(f'  Cold-start (skipped)  : {cold_start_count:,}')
print(f'  Fit errors            : {error_count:,}')
print(f'  Successful forecasts  : {n_groups - cold_start_count - error_count:,}')

# FIX 4: Error report 
if error_log:
    print(f'\nError detail ({len(error_log)} total):')
    error_types = pd.Series([e['error'][:80] for e in error_log]).value_counts()
    print(error_types.head(10))
    err_path = os.path.join(ARTIFACT_DIR, 'prophet_fit_errors.csv')
    pd.DataFrame(error_log).to_csv(err_path, index=False)
    print(f'Full error log saved to {err_path}')
else:
    print('\nNo fit errors.')

# Save forecasts keyed by (sim_id, cycle_num_str) for fast lookup in later cells
fc_lookup = {(r['sim_id'], r['billing_cycle_num']): r for r in results}
print('Forecast results stored in fc_lookup.')

FULL MODE: running on all SIMs.


13:26:46 - cmdstanpy - INFO - Chain [1] start processing


Total (sim_id, billing_cycle_num) groups: 3,000
Timing one Prophet fit to estimate total runtime ...


13:26:49 - cmdstanpy - INFO - Chain [1] done processing
13:26:49 - cmdstanpy - INFO - Chain [1] start processing


  Single fit time : 3.38 s
  Estimated total : 10146 s  (169.1 min)
  [!] > 30 min -- consider keeping SUBSAMPLE_SIMS set for exploration.

Starting forecast loop ...


...

  [2900/3000]  elapsed=5305s  rate=0.55/s  ETA=183s


14:55:14 - cmdstanpy - INFO - Chain [1] done processing
14:55:15 - cmdstanpy - INFO - Chain [1] start processing
14:55:17 - cmdstanpy - INFO - Chain [1] done processing
14:55:17 - cmdstanpy - INFO - Chain [1] start processing
14:55:17 - cmdstanpy - INFO - Chain [1] done processing
14:55:18 - cmdstanpy - INFO - Chain [1] start processing
14:55:18 - cmdstanpy - INFO - Chain [1] done processing
14:55:18 - cmdstanpy - INFO - Chain [1] start processing
14:55:19 - cmdstanpy - INFO - Chain [1] done processing
14:55:19 - cmdstanpy - INFO - Chain [1] start processing
14:55:21 - cmdstanpy - INFO - Chain [1] done processing
14:55:21 - cmdstanpy - INFO - Chain [1] start processing
14:55:25 - cmdstanpy - INFO - Chain [1] done processing
14:55:25 - cmdstanpy - INFO - Chain [1] start processing
14:55:26 - cmdstanpy - INFO - Chain [1] done processing
14:55:26 - cmdstanpy - INFO - Chain [1] start processing
14:55:28 - cmdstanpy - INFO - Chain [1] done processing
14:55:28 - cmdstanpy - INFO - Chain [1] 

  [2950/3000]  elapsed=5398s  rate=0.55/s  ETA=91s


14:56:49 - cmdstanpy - INFO - Chain [1] done processing
14:56:49 - cmdstanpy - INFO - Chain [1] start processing
14:56:51 - cmdstanpy - INFO - Chain [1] done processing
14:56:51 - cmdstanpy - INFO - Chain [1] start processing
14:56:52 - cmdstanpy - INFO - Chain [1] done processing
14:56:52 - cmdstanpy - INFO - Chain [1] start processing
14:56:53 - cmdstanpy - INFO - Chain [1] done processing
14:56:53 - cmdstanpy - INFO - Chain [1] start processing
14:56:54 - cmdstanpy - INFO - Chain [1] done processing
14:56:54 - cmdstanpy - INFO - Chain [1] start processing
14:56:55 - cmdstanpy - INFO - Chain [1] done processing
14:56:56 - cmdstanpy - INFO - Chain [1] start processing
14:56:57 - cmdstanpy - INFO - Chain [1] done processing
14:56:57 - cmdstanpy - INFO - Chain [1] start processing
14:56:59 - cmdstanpy - INFO - Chain [1] done processing
14:56:59 - cmdstanpy - INFO - Chain [1] start processing
14:57:00 - cmdstanpy - INFO - Chain [1] done processing
14:57:00 - cmdstanpy - INFO - Chain [1] 

  [3000/3000]  elapsed=5482s  rate=0.55/s  ETA=0s

Loop complete in 5482 s (91.4 min)
  Cold-start (skipped)  : 0
  Fit errors            : 0
  Successful forecasts  : 3,000

No fit errors.
Forecast results stored in fc_lookup.


## Cell 10 -- Cold-Start Caveat: Ghost Usage & Inactive Billing

**First join of labels** -- used only to characterise forecast quality for
SIMs with flat near-zero usage, not to influence any forecast decision.

Ghost usage and inactive billing SIMs have very low or zero `total_mb`.
Prophet's trend extrapolation has little signal to work with for these SIMs,
resulting in low `projected_overage_mb` (correct -- they don't deplete their
bundle, which is the anomaly, not overuse). This is a **known limitation**,
not a bug: Prophet is the right tool for detecting early *depletion*, not
for detecting *under-usage* anomalies.

In [10]:
# Cell 10: Ghost/Inactive Caveat Report (labels joined here for analysis only)

GHOST_CATS = {'ghost_usage', 'inactive_billing'}
ghost_lbl  = labels_raw[labels_raw['category'].isin(GHOST_CATS)].copy()

print(f'Ghost/inactive episodes: {len(ghost_lbl):,}')
print(f'  ghost_usage           : {(ghost_lbl.category=="ghost_usage").sum()}')
print(f'  inactive_billing      : {(ghost_lbl.category=="inactive_billing").sum()}')

ghost_stats = []
for _, row in ghost_lbl.iterrows():
    sid = row['sim_id']
    # Find which fixed 30-day cycles overlap this episode
    sim_agg = agg[agg.sim_id == sid]
    ep_agg  = sim_agg[(sim_agg.date >= row['start_date']) &
                       (sim_agg.date <= row['end_date'])]
    cycles  = ep_agg['billing_cycle_num'].unique() if len(ep_agg) else []
    for cyc in cycles:
        key = (sid, str(cyc))
        if key in fc_lookup:
            r = fc_lookup[key]
            ghost_stats.append({
                'sim_id':               sid,
                'category':             row['category'],
                'cycle':                str(cyc),
                'projected_overage_mb': r['projected_overage_mb'],
                'depletion_flag':       r['depletion_flag'],
                'predicted_final_cum':  r['predicted_final_cum'],
            })

if ghost_stats:
    gdf = pd.DataFrame(ghost_stats)
    print(f'\n[!]  KNOWN COLD-START CAVEAT -- ghost_usage / inactive_billing SIMs')
    print(f'   Prophet is designed to detect EARLY DEPLETION (overuse), not under-usage.')
    print(f'   These SIMs have near-zero usage so Prophet correctly predicts low')
    print(f'   cumulative totals, which will NOT trigger depletion_flag=True.')
    print(f'   This is expected behaviour, not a model failure.')
    print()
    print(f'   Affected (sim_id, cycle) pairs: {len(gdf):,}')
    print(f'   depletion_flag=True  (should be low): ',
          f'{gdf["depletion_flag"].sum()}')
    print(f'   Mean projected_overage_mb           : ',
          f'{gdf["projected_overage_mb"].mean():.1f} MB')
    print(f'   Mean predicted_final_cum            : ',
          f'{gdf["predicted_final_cum"].mean():.1f} MB')
    print()
    print('   -> Downstream anomaly detection for ghost_usage / inactive_billing')
    print('     should rely on Isolation Forest or cohort z-score, NOT Prophet.')
else:
    print('No ghost/inactive episodes found in fc_lookup.')

print('\n Cold-start caveat documented.')

Ghost/inactive episodes: 64
  ghost_usage           : 43
  inactive_billing      : 21

[!]  KNOWN COLD-START CAVEAT -- ghost_usage / inactive_billing SIMs
   Prophet is designed to detect EARLY DEPLETION (overuse), not under-usage.
   These SIMs have near-zero usage so Prophet correctly predicts low
   cumulative totals, which will NOT trigger depletion_flag=True.
   This is expected behaviour, not a model failure.

   Affected (sim_id, cycle) pairs: 82
   depletion_flag=True  (should be low):  0
   Mean projected_overage_mb           :  -39604.6 MB
   Mean predicted_final_cum            :  10346.6 MB

   -> Downstream anomaly detection for ghost_usage / inactive_billing
     should rely on Isolation Forest or cohort z-score, NOT Prophet.

 Cold-start caveat documented.


## Cell 11 — Build Feature Table (Tier Recommendation)

Builds `features_df` from `fc_lookup` (one row per `sim_id`, `billing_cycle_num`).
Translates `predicted_final_cum` into a tier recommendation by comparing the
forecasted GB against the sorted tier limits.

The output is saved to `data/prophet_features.parquet`.

In [11]:
# Cell 11: Build Feature Table

rows = []

for (sim_id, cycle), grp in agg.groupby(['sim_id', 'billing_cycle_num']):
    tier = grp['tier'].iloc[0]
    date = grp['date'].max()  # for joining with labels later

    key = (sim_id, str(cycle))
    r   = fc_lookup.get(key, {})

    pred_final = r.get('predicted_final_cum',  float('nan'))
    overage_mb = r.get('projected_overage_mb', float('nan'))
    days_dep   = r.get('days_to_depletion',    float('nan'))

    rows.append({
        'sim_id':               sim_id,
        'billing_cycle_num':    str(cycle),
        'date':                 date,
        'tier':                 tier,
        'predicted_final_cum':  pred_final,
        'projected_overage_mb': overage_mb,
        'days_to_depletion':    days_dep,
    })

features_df = pd.DataFrame(rows)
features_df['date'] = pd.to_datetime(features_df['date'])

print(f'features_df shape : {features_df.shape}')
cold_start_rows = features_df['predicted_final_cum'].isna().sum()
print(f'Rows with predicted_final_cum = NaN (cold-start/error): {cold_start_rows:,}')

TIER_BUNDLE_GB_SORTED = [('support', 20), ('professional', 40),
                         ('manager', 60), ('field', 100)]
TIER_RANK = {t: i for i, (t, _) in enumerate(TIER_BUNDLE_GB_SORTED)}

def recommend_tier(predicted_final_cum_mb):
    if pd.isna(predicted_final_cum_mb):
        return None
    predicted_gb = predicted_final_cum_mb / 1024
    for tier_name, bundle_gb in TIER_BUNDLE_GB_SORTED:
        if predicted_gb <= bundle_gb:
            return tier_name
    return 'field'

features_df['recommended_tier'] = features_df['predicted_final_cum'].apply(recommend_tier)
features_df['current_tier_rank'] = features_df['tier'].map(TIER_RANK)
features_df['recommended_tier_rank'] = features_df['recommended_tier'].map(TIER_RANK)
features_df['recommendation'] = np.select(
    [features_df['recommended_tier_rank'] < features_df['current_tier_rank'],
     features_df['recommended_tier_rank'] > features_df['current_tier_rank']],
    ['downgrade', 'upgrade'],
    default='no_change'
)
features_df.loc[features_df['recommended_tier'].isna(), 'recommendation'] = 'insufficient_history'

print('\nRecommendation value_counts:')
print(features_df['recommendation'].value_counts())
print('\nRecommended_tier value_counts:')
print(features_df['recommended_tier'].value_counts())

# Sanity: no label columns
for lc in ['is_anomalous', 'misuse_category', 'category']:
    assert lc not in features_df.columns, f'Label {lc!r} leaked into features!'
print('\n[OK] No label columns in features_df.')

out_path = os.path.join(DATA_DIR, 'prophet_features.parquet')
features_df.to_parquet(out_path, index=False)
print(f'Saved -> {out_path}')

features_df shape : (3000, 7)
Rows with predicted_final_cum = NaN (cold-start/error): 0

Recommendation value_counts:
recommendation
downgrade    2378
no_change     610
upgrade        12
Name: count, dtype: int64

Recommended_tier value_counts:
recommended_tier
support         2355
professional     588
manager           52
field              5
Name: count, dtype: int64

[OK] No label columns in features_df.
Saved -> data\prophet_features.parquet


## Cell 12 — Evaluation with Misuse-Flag Override

Joins labels ONLY here for reporting. Overrides clean `downgrade` recommendations
if there is an active misuse flag, changing the action to `HOLD`.

In [12]:
# Cell 12: Evaluation with misuse-flag override

# Expand labels to daily rows to allow date-based join
label_days = []
for _, row in labels_raw.iterrows():
    dates = pd.date_range(row['start_date'], row['end_date'])
    for d in dates:
        label_days.append({
            'sim_id': row['sim_id'],
            'date': d,
            'misuse_category': row['category']
        })
labels_df = pd.DataFrame(label_days)
labels_df['date'] = pd.to_datetime(labels_df['date'])

features_df = features_df.merge(
    labels_df[['sim_id', 'date', 'misuse_category']],
    on=['sim_id', 'date'], how='left'
)
features_df['misuse_flag_active'] = features_df['misuse_category'].notna()

features_df['final_action'] = np.where(
    (features_df['recommendation'] == 'downgrade') & features_df['misuse_flag_active'],
    'HOLD - review before downgrading (active misuse flag)',
    features_df['recommendation']
)

print('features_df["recommendation"] value_counts:')
print(features_df['recommendation'].value_counts())
print('\nfeatures_df["final_action"] value_counts:')
print(features_df['final_action'].value_counts())

print('\nFinal action breakdown by misuse_category:')
crosstab = pd.crosstab(features_df['misuse_category'].fillna('None'), features_df['final_action'])
print(crosstab)

holds = features_df[features_df['final_action'].str.startswith('HOLD', na=False)]
if not holds.empty:
    print(f'\nRows HELD ({len(holds)}):')
    print(holds[['sim_id', 'date', 'misuse_category', 'tier', 'recommended_tier']].head(20).to_string())
else:
    print('\nNo rows were HELD.')

# Re-save with eval
eval_path = os.path.join(DATA_DIR, 'prophet_features_with_eval.parquet')
features_df.to_parquet(eval_path, index=False)
print(f'\nSaved evaluated features -> {eval_path}')
print('[OK] Evaluation complete.')

features_df["recommendation"] value_counts:
recommendation
downgrade    2378
no_change     610
upgrade        12
Name: count, dtype: int64

features_df["final_action"] value_counts:
final_action
downgrade                                                2327
no_change                                                 610
HOLD - review before downgrading (active misuse flag)      51
upgrade                                                    12
Name: count, dtype: int64

Final action breakdown by misuse_category:
final_action          HOLD - review before downgrading (active misuse flag)  \
misuse_category                                                               
None                                                                  0       
data_reselling                                                        9       
ghost_usage                                                          21       
inactive_billing                                                      7       
unauthorized_

## Cell 13 — Example Plots and MLflow Logging

Plots three examples:
1. A SIM getting an upgrade recommendation.
2. A clean downgrade (no misuse flag).
3. A downgrade candidate that got HELD due to an active misuse flag.

In [13]:
# Cell 13: Example Plots

def find_example_sim(action, features_df):
    sub = features_df[features_df['final_action'] == action]
    if len(sub) == 0:
        return None, None
    row = sub.iloc[0]
    return row['sim_id'], row['billing_cycle_num']

# 1. upgrade
upg_sid, upg_cycle = find_example_sim('upgrade', features_df)
# 2. downgrade (clean)
dwn_sid, dwn_cycle = find_example_sim('downgrade', features_df)
# 3. HOLD
hld_sid, hld_cycle = find_example_sim('HOLD - review before downgrading (active misuse flag)', features_df)

print(f'Upgrade SIM       : sim_id={upg_sid}, cycle={upg_cycle}')
print(f'Downgrade (clean) : sim_id={dwn_sid}, cycle={dwn_cycle}')
print(f'HOLD override     : sim_id={hld_sid}, cycle={hld_cycle}')

# -- Plot function --
def plot_sim_cycle(ax, sim_id, cycle_str, features_df, agg_df, fc_lookup):
    sim_cyc = agg_df[(agg_df.sim_id == sim_id) &
                      (agg_df.billing_cycle_num.astype(str) == cycle_str)]
    sim_cyc = sim_cyc.sort_values('date')
    bundle_mb = float(sim_cyc['bundle_mb'].iloc[0])
    cum_actual = sim_cyc['total_mb'].cumsum().values

    ax.plot(sim_cyc['date'], cum_actual, 'o-', color='#1e3a5f',
            ms=3, lw=1.5, label='Actual cumulative MB')

    key = (sim_id, cycle_str)
    fc  = fc_lookup.get(key, {})
    fdf = fc.get('forecast_df', None)

    if fdf is not None:
        fdf = fdf.copy()
        fdf['ds'] = pd.to_datetime(fdf['ds'])
        last_obs = fc.get('last_obs_date', sim_cyc['date'].iloc[-1])
        future   = fdf[fdf['ds'] > last_obs]
        if len(future) > 0:
            ax.plot(future['ds'], future['yhat'], '--', color='#e07b39',
                    lw=1.5, label='Prophet forecast')
            ax.fill_between(future['ds'], future['yhat_lower'],
                            future['yhat_upper'], alpha=0.15, color='#e07b39')
    
    # Title info from features_df
    feat_row = features_df[(features_df.sim_id == sim_id) & 
                           (features_df.billing_cycle_num == cycle_str)]
    if len(feat_row) > 0:
        f = feat_row.iloc[0]
        title = f"SIM {sim_id} | {f['tier']} -> {f['recommended_tier']}\nAction: {f['final_action']}"
    else:
        title = f"SIM {sim_id} (not in features)"

    ax.axhline(bundle_mb, color='#c0392b', lw=1.2, ls=':',
               label=f'Bundle limit ({bundle_mb/1024:.0f} GB)')
    ax.set_title(title, fontsize=9, fontweight='bold')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
    ax.tick_params(axis='x', labelsize=7, rotation=30)
    ax.set_ylabel('Cumulative MB', fontsize=8)
    ax.legend(fontsize=7, loc='upper left')
    ax.grid(True, alpha=0.25)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Prophet Burn-Rate Forecasting -- Tier Recommendations', fontsize=13, fontweight='bold')

if upg_sid is not None:
    plot_sim_cycle(axes[0], upg_sid, upg_cycle, features_df, agg, fc_lookup)
else:
    axes[0].set_title('No upgrade example found')

if dwn_sid is not None:
    plot_sim_cycle(axes[1], dwn_sid, dwn_cycle, features_df, agg, fc_lookup)
else:
    axes[1].set_title('No downgrade example found')

if hld_sid is not None:
    plot_sim_cycle(axes[2], hld_sid, hld_cycle, features_df, agg, fc_lookup)
else:
    axes[2].set_title('No HOLD example found')

fig.tight_layout()
PLOT_PATH = os.path.join(ARTIFACT_DIR, 'prophet_tier_examples.png')
fig.savefig(PLOT_PATH, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved -> {PLOT_PATH}')

# MLflow logging
if mlflow.active_run() is not None:
    mlflow.end_run()

with mlflow.start_run(run_name='prophet-burn-rate-forecast') as run:
    mlflow.log_param('min_history_days',    MIN_HISTORY_DAYS)
    mlflow.log_param('n_sim_cycle_groups',  n_groups)
    mlflow.log_param('cold_start_count',    cold_start_count)
    mlflow.log_param('fit_errors',          error_count)
    mlflow.log_param('single_fit_time_s',   round(single_fit_s, 3))
    mlflow.log_param('subsample_sims',      str(SUBSAMPLE_SIMS))
    mlflow.log_param('cycle_type',          'fixed_30_day')
    
    mlflow.log_metric('pct_upgrade',       (features_df['recommendation']=='upgrade').mean())
    mlflow.log_metric('pct_downgrade',     (features_df['recommendation']=='downgrade').mean())
    mlflow.log_metric('pct_no_change',     (features_df['recommendation']=='no_change').mean())
    mlflow.log_metric('pct_hold_override', features_df['final_action'].str.startswith('HOLD', na=False).mean())
    
    mlflow.log_artifact(PLOT_PATH)
    mlflow.log_artifact(os.path.join(DATA_DIR, 'prophet_features.parquet'))
    if error_log:
        err_path = os.path.join(ARTIFACT_DIR, 'prophet_fit_errors.csv')
        if os.path.exists(err_path):
            mlflow.log_artifact(err_path)
    print(f'MLflow run logged: {run.info.run_id}')

print('\n[OK] prophet_burn_rate.ipynb complete.')

Upgrade SIM       : sim_id=78, cycle=0
Downgrade (clean) : sim_id=0, cycle=0
HOLD override     : sim_id=21, cycle=1
Saved -> artifacts\prophet_tier_examples.png
MLflow run logged: 385d6bced50e403fb25f093844ea2513

[OK] prophet_burn_rate.ipynb complete.
